## Camada Silver —> Limpeza e Enriquecimento

Este notebook aplica transformações sobre os dados da camada Bronze: conversão de tipos,
remoção de registros inativos e cancelados, junção entre as tabelas e cálculo do valor total
por item considerando descontos aplicados.

**Fonte:** `/Volumes/dados/default/desafio_03/bronze/`  
**Destino:** `/Volumes/dados/default/desafio_03/silver/pedidos_enriquecidos`

In [0]:
# 02_silver.py
from pyspark.sql import functions as F

BRONZE_PATH = "/Volumes/dados/default/desafio_03/bronze"
SILVER_PATH = "/Volumes/dados/default/desafio_03/silver"

# 1. Leitura das tabelas Bronze 

clientes     = spark.read.format("delta").load(f"{BRONZE_PATH}/clientes")
produtos     = spark.read.format("delta").load(f"{BRONZE_PATH}/produtos")
vendedores   = spark.read.format("delta").load(f"{BRONZE_PATH}/vendedores")
pedidos      = spark.read.format("delta").load(f"{BRONZE_PATH}/pedidos")
itens_pedido = spark.read.format("delta").load(f"{BRONZE_PATH}/itens_pedido")

# 2. Conversão de datas

pedidos = pedidos.withColumn("data_pedido", F.to_date("data_pedido"))

# 3. Filtros de limpeza

clientes   = clientes.filter(F.col("status") != "inativo")
produtos   = produtos.filter(F.col("status") != "inativo")
vendedores = vendedores.filter(F.col("status") != "inativo")
pedidos    = pedidos.filter(F.col("status_pedido") != "cancelado")

# ── 4. Joins + remoção de colunas duplicadas ─────────────────────────────────

itens_pedido = itens_pedido.withColumn("id_produto", F.expr("try_cast(id_produto as BIGINT)"))

df = itens_pedido \
    .join(pedidos,    "id_pedido") \
    .join(clientes,   "id_cliente") \
    .join(vendedores, "id_vendedor") \
    .join(produtos,   "id_produto") \
    .drop(itens_pedido["preco_unitario"]) \
    .drop(produtos["status"]) \
    .drop(vendedores["status"]) \
    .drop(clientes["estado"]) \
    .drop(produtos["marca"])

# 5. Cálculo do valor total por item 

df = df.withColumn(
    "valor_total_item",
    F.col("quantidade") * F.col("preco_unitario") * (1 - F.col("desconto"))
)

# 6. Salvando na camada Silver 
df.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/pedidos_enriquecidos")

print("Silver concluído")
display(df)